In [2]:
import feedparser
import requests
import os
import time
from google import genai

In [ ]:
#默认获取最新一期播客，如果你想处理指定文件，可以修改逻辑

def get_latest_podcast_audio(rss_url):
    """解析 RSS 链接并获取最新一期播客的音频 URL 和标题"""
    print("正在解析 RSS 链接...")
    feed = feedparser.parse(rss_url)
    
    if not feed.entries:
        raise Exception("未找到任何播客内容，请检查 RSS 链接是否正确！")

    latest_episode = feed.entries[0]
    title = latest_episode.title

    # 遍历 links 寻找音频文件链接 (通常 rel 为 enclosure)
    audio_url = None
    for link in latest_episode.links:
        if link.rel == 'enclosure' and 'audio' in link.type:
            audio_url = link.href
            break

    if not audio_url:
        raise Exception("在最新一期节目中未找到音频下载链接！")

    print(f"找到最新一期播客: {title}")
    return title, audio_url

def download_audio(audio_url, filename="temp_podcast.mp3"):
    """下载音频文件到本地"""
    print(f"正在下载音频 (可能需要几分钟，取决于文件大小): {audio_url}")
    response = requests.get(audio_url, stream=True)
    response.raise_for_status()
    
    with open(filename, 'wb') as f:
        # 分块下载以避免内存占用过大
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            
    print("✅ 音频下载完成！")
    return filename

def process_with_gemini(audio_path):
    """上传音频到 Gemini API 并获取逐字稿和翻译"""
    print("正在将音频上传至 Google 服务器 (Gemini File API)...")
    # 播客文件通常较大，必须使用 File API 上传
    client = genai.Client()

    # 正确的调用方式
    audio_file = client.files.upload(file = audio_path)
    #audio_file = genai.upload_file(path=audio_path)

    # 上传大文件后，Gemini 需要一点时间处理音频，需轮询等待状态变为 ACTIVE
    while audio_file.state.name == "PROCESSING":
        print("等待 Gemini 处理音频流...")
        time.sleep(10)
        # 刷新文件状态
        audio_file = genai.get_file(audio_file.name)

    if audio_file.state.name == "FAILED":
        raise Exception("Gemini 音频处理失败！")

    print("✅ 音频准备就绪，正在请求 AI 进行听写和翻译 (视音频长度可能需要 1-3 分钟)...")
    
    # 使用 Gemini 1.5 Pro (对于长音频转录和翻译的质量最好)
    # 如果想追求速度，可以改为 "gemini-1.5-flash"
    model = "gemini-2.5-flash"

    # 设计 Prompt 让模型一次性输出英文原文和中文翻译
    prompt = """
    Please listen to this English podcast audio carefully.
    
    Your task:
    1. Transcribe the entire audio into English text.
    2. Translate the transcribed English text into natural, fluent Simplified Chinese.

    Please output EXACTLY in the following format:

    [English Transcript]
    (Insert the full English transcript here)

    [Chinese Translation]
    (Insert the full Chinese translation here)
    """

    # 调用模型生成内容
    response = client.models.generate_content(model=model, contents = [audio_file, prompt])

    # 养成好习惯：处理完毕后从 Google 服务器删除临时文件以节省空间
    print("正在清理云端临时音频文件...")
    #genai.delete_file(audio_file.name)
    client.files.delete(name=audio_file.name)
    print(response.text)

    return response.text

def main():
    rss_url = input("请输入英文播客的 RSS 链接: ").strip()

    try:
        # 1. 获取并下载音频
        title, audio_url = get_latest_podcast_audio(rss_url)
        audio_filename = "temp_podcast.mp3"
        download_audio(audio_url, audio_filename)

        # 2. 调用 Gemini API
        result_text = process_with_gemini(audio_filename)

        # 3. 将结果保存到文本文件
        # 清理标题中可能导致文件路径错误的特殊字符
        safe_title = "".join([c for c in title if c.isalpha() or c.isdigit() or c==' ']).rstrip()
        output_filename = f"{safe_title}_transcript.txt"
        
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write(result_text)

        print(f"\n🎉 处理成功！文稿和翻译已保存至当前目录: {output_filename}")

    except Exception as e:
        print(f"\n❌ 发生错误: {e}")
        
    finally:
        # 4. 删除本地下载的临时音频文件
        if os.path.exists("temp_podcast.mp3"):
            os.remove("temp_podcast.mp3")
            print("本地临时文件已清理。")

if __name__ == "__main__":
    main()

正在解析 RSS 链接...
找到最新一期播客: NPR News: 04-01-2026 3AM EDT
正在下载音频 (可能需要几分钟，取决于文件大小): https://prfx.byspotify.com/e/play.podtrac.com/npr-500005/npr.simplecastaudio.com/368f8510-314e-46f1-8d26-5b87ed6ab6eb/episodes/95730681-5889-4a3a-962d-f49f577d0a15/audio/128/default.mp3?awCollectionId=368f8510-314e-46f1-8d26-5b87ed6ab6eb&awEpisodeId=95730681-5889-4a3a-962d-f49f577d0a15&feed=O9WlY7a5&t=podcast&e=nx-s1-20260401-0300-long&p=500005&d=280&size=4480567
✅ 音频下载完成！
正在将音频上传至 Google 服务器 (Gemini File API)...
✅ 音频准备就绪，正在请求 AI 进行听写和翻译 (视音频长度可能需要 1-3 分钟)...
正在清理云端临时音频文件...
[English Transcript]
Live from NPR News, I'm Giles Snyder. President Trump is planning to deliver an address to the nation tonight. The White House says he'll give an update on the Iran War. Trump's address comes as the Strait of Hormuz remains largely shut down, sending oil and gas prices sharply higher. NPR's Stephen reports. Nationally, a gallon of regular gas at the pump now costs on average more than $4. Not really all that much hi

In [ ]:
print()